<h1>Get conserved clusters intra species<h1/>

<h2>Import librairies

In [2]:
import pymongo
from pymongo import mongo_client
import pandas as pd


<h2>Connect to MongoDB

In [3]:
from pymongo import mongo_client


MongoClient = pymongo.MongoClient("localhost", 27017)


clusters_db = "LPM_w15_ncc"   #nuclear proteome


familly = clusters_db

conserved_cluster_intra = "Conserved_cluster_intra"

ConcervedINTRA = MongoClient[conserved_cluster_intra]
clusterDB = MongoClient[clusters_db]
#clusterMDB = MongoClient[clusters_m_db]


In [9]:
species = clusterDB.list_collection_names()  # activate when screening among NM


for specie in species:
    print(specie)
    collections = clusterDB.get_collection(specie)
    documents = collections.find()
    for item in documents:
        #print(item)
        cluster = item["cluster"]

        cnt = collections.count_documents(
            {"cluster": cluster, "id": {"$ne": item["id"]}}
        )

        if cnt > 0:
            print(cnt)
            collec = ConcervedINTRA.get_collection(familly)
            if collec.find_one({"specie": specie}):
                if collec.find_one({"$and": [
                    {"specie": specie},
                    {f"clusters.{cluster}": {"$exists": True}}]}):
                    print("cluster exist")
                    collec.update_one({"$and": [
                    {"specie": specie},
                    {f"clusters.{cluster}": {"$exists": True}}]},
                        {"$push": {f"clusters.{cluster}.ids": item["id"]}},
                    )
                else:
                    print("insert new cluster")
                    collec.update_one(
                        {"specie": specie},
                        {"$set": {f"clusters.{cluster}": {"ids": [item["id"]]}}},
                    )
            else:
                print("new doc")
                entry = {"specie": specie, "clusters": {cluster: {"ids": [item["id"]]}}}
                collec.insert_one(entry)
        